# PRISM: Temporal Transformer Training (Sprint 4)
This notebook trains the Temporal Transformer on Colab GPU to predict disease trajectory.

**Prerequisites:**
1. `prism-colab/` folder on Google Drive (with the updated `models/temporal_transformer/` code)
2. No additional data upload needed -- synthetic data is generated in this notebook

**Model:**
- Lightweight 3-layer encoder-only Transformer (407K params)
- Input: 30 days x 5 daily cough statistics
- Output: 4 trajectory classes (Stable, Improving, Increasing, Abnormal)

**Output:**
- `temporal_transformer_best.pt` -- best model checkpoint

In [ ]:
# === Cell 1: Setup Environment and Mount Drive ===
!pip install -q loguru rich scikit-learn pyyaml tqdm

import os

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Symlink the PRISM code into the Colab working directory
if not os.path.exists('/content/prism'):
    os.symlink('/content/drive/MyDrive/prism-colab', '/content/prism')
if not os.path.exists('/content/models'):
    os.symlink('/content/drive/MyDrive/prism-colab/models', '/content/models')

print('\nCode setup complete!')

In [ ]:
# === Cell 2: Verify GPU ===
import torch

if torch.cuda.is_available():
    print(f'GPU Active: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('No GPU found -- this model is small enough for CPU too.')
    print('For faster training: Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# === Cell 3: Generate Synthetic Temporal Data ===
import os

os.chdir('/content/prism')

# Create the temporal data directory on local Colab disk (fast I/O)
!mkdir -p /content/temporal_data

!python -m models.temporal_transformer.generate_temporal_data \
    --patients-per-class 500 \
    --seed 42 \
    --output-dir /content/temporal_data

# Verify files were created
!ls -la /content/temporal_data/

In [ ]:
# === Cell 4: Dry Run (Sanity Check) ===
import os

os.chdir('/content/prism')

!python -m models.temporal_transformer.run_training \
    --data-dir /content/temporal_data \
    --dry-run

In [ ]:
# === Cell 5: Train the Temporal Transformer ===
import os

os.chdir('/content/prism')

!python -m models.temporal_transformer.run_training \
    --data-dir /content/temporal_data \
    --epochs 100 \
    --batch-size 16

In [ ]:
# === Cell 6: Evaluate on Test Set ===
import os

os.chdir('/content/prism')

# Run evaluation
import sys

sys.path.insert(0, '/content/prism')

from models.temporal_transformer.evaluate import run_evaluation

metrics = run_evaluation(
    checkpoint_path='models/checkpoints/temporal_transformer_best.pt',
    data_dir='/content/temporal_data',
    output_path='/content/temporal_eval.json',
)

print(f"\nTest Accuracy: {metrics['accuracy']:.4f}")
print(f"Test Macro F1: {metrics['macro_f1']:.4f}")

In [ ]:
# === Cell 7: Save Checkpoint and Results to Google Drive ===
import os
import shutil

drive_checkpoints = '/content/drive/MyDrive/prism-colab/checkpoints'
os.makedirs(drive_checkpoints, exist_ok=True)

# Copy the best checkpoint
src_checkpoint = '/content/prism/models/checkpoints/temporal_transformer_best.pt'
if os.path.exists(src_checkpoint):
    shutil.copy2(src_checkpoint, drive_checkpoints)
    print(f'Checkpoint saved to: {drive_checkpoints}/temporal_transformer_best.pt')
else:
    print('ERROR: Checkpoint not found. Did training complete successfully?')

# Copy evaluation results
if os.path.exists('/content/temporal_eval.json'):
    shutil.copy2('/content/temporal_eval.json', drive_checkpoints)
    print(f'Eval results saved to: {drive_checkpoints}/temporal_eval.json')

# Also copy the generated temporal data for reproducibility
drive_data = '/content/drive/MyDrive/prism-colab/datasets/temporal'
os.makedirs(drive_data, exist_ok=True)
for f in ['temporal_train.csv', 'temporal_val.csv', 'temporal_test.csv']:
    shutil.copy2(f'/content/temporal_data/{f}', drive_data)
print(f'Temporal data saved to: {drive_data}/')

print('\nAll done! You can now download the checkpoint to your local machine:')
print('   -> Place it in: PRISM/models/checkpoints/temporal_transformer_best.pt')